In [1]:
import langchain_openai

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [3]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o-mini")

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [5]:
prompt = ChatPromptTemplate.from_template("tell me a curious fact about {politician}")

In [6]:
chain1=(prompt)

In [7]:
result=chain1.invoke("Trump")
result

ChatPromptValue(messages=[HumanMessage(content='tell me a curious fact about Trump')])

In [8]:
chain2=(prompt|chatModel)

In [9]:
result=chain2.invoke("Trump")
result

AIMessage(content='A curious fact about Donald Trump is that before he entered politics, he was an accomplished television personality. He gained widespread fame as the host of the reality TV show "The Apprentice," which premiered in 2004. The show featured contestants competing for a job within Trump\'s organization, and his catchphrase "You\'re fired!" became a cultural phenomenon. Trump\'s role on the show significantly boosted his public profile and contributed to his eventual run for the presidency in 2016.', response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 14, 'total_tokens': 107, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_dbaca60df0', 'finish_reason': 'stop', 'logprobs': None}, id='run-906c9c1a-eeb5-42a8-8c7f-ae3c689bd5f2-0', 

In [10]:
chain3=(prompt|chatModel|StrOutputParser())
result=chain3.invoke("Trump")
result

'One curious fact about Donald Trump is that before entering politics, he was known for his appearances on various television shows and reality TV. He gained widespread fame as the host of "The Apprentice," a reality competition series that premiered in 2004. The show\'s success helped solidify his public persona as a businessman and entrepreneur, showcasing his catchphrase "You\'re fired!" which became iconic. Interestingly, Trump\'s foray into reality television played a significant role in shaping the image he presented during his political career.'

In [11]:
from langchain_core.runnables import RunnablePassthrough

chain = RunnablePassthrough()

chain.invoke("selamlar")

'selamlar'

In [12]:
def rusismi(text):
    return text+"ovic"


In [13]:
print(rusismi("ibrahim"))

ibrahimovic


In [14]:
from langchain_core.runnables import RunnableParallel

chain = RunnableParallel(
{
    "isim": RunnablePassthrough(),
    "rusyadaki adı": rusismi
}
)

chain.invoke("ibrahim")


{'isim': 'ibrahim', 'rusyadaki adı': 'ibrahimovic'}

In [15]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("./LOR.txt", encoding="utf-8")

loaded_data = loader.load()

In [16]:
from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)
texts = text_splitter.create_documents([loaded_data[0].page_content])
text_contents = []
for item in texts:
    text_contents.append(item.page_content)

Created a chunk of size 1855, which is longer than the specified 1000
Created a chunk of size 1787, which is longer than the specified 1000
Created a chunk of size 1861, which is longer than the specified 1000
Created a chunk of size 1292, which is longer than the specified 1000
Created a chunk of size 1371, which is longer than the specified 1000
Created a chunk of size 1298, which is longer than the specified 1000
Created a chunk of size 1076, which is longer than the specified 1000
Created a chunk of size 1740, which is longer than the specified 1000
Created a chunk of size 1041, which is longer than the specified 1000
Created a chunk of size 1512, which is longer than the specified 1000
Created a chunk of size 1544, which is longer than the specified 1000
Created a chunk of size 1505, which is longer than the specified 1000


In [17]:
from langchain_openai import OpenAIEmbeddings
embeddings_model = OpenAIEmbeddings()

In [18]:
from langchain_chroma import Chroma
vector_db = Chroma.from_documents(texts, OpenAIEmbeddings())
retriever = vector_db.as_retriever()

In [19]:
template = """Sana verdiğim  bilgilere dayanarak soruyu kısaca cevapla:
bilgi:
{context}

Soru: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [20]:
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

In [21]:
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | chatModel
    | StrOutputParser()
)

In [22]:
response = chain.invoke("who made the one ring")
response

'The One Ring was created by the Dark Lord Sauron.'